# Dimensionless Kronig--Penney delta-comb bands

**Prerequisite:** [Kronig--Penney delta-comb dispersion](../../../../docs/lecturenotes/solidstate/electronic-structure/kronig-penney/delta-comb/index.md)

This laboratory preserves the exploratory Kronig--Penney band-structure graph while replacing tolerance-based matching on a two-dimensional mesh with direct evaluation of the allowed dispersion branches.

## Learning objectives

1. Evaluate the dimensionless delta-comb dispersion relation.
2. Identify allowed energies from the real Bloch-phase condition.
3. Plot the symmetric first-Brillouin-zone branches.
4. Distinguish the dimensionless barrier parameter from an energy-valued barrier height.

The selected barrier strength is illustrative and is not fitted to a material.


## Dimensionless model

For lattice period $a$, free-region wave number $q$, and $z=qa$, write

$$
\cos(ka)=\cos z+P\frac{\sin z}{z},
$$

where $P\geq0$ is a dimensionless delta-comb strength. Real Bloch phases exist only when the right-hand side lies in $[-1,1]$. The plotted reduced energy is

$$
\frac{E}{E_a}=z^2,
\qquad
E_a=\frac{\hbar^2}{2ma^2}.
$$

This convention avoids assigning energy units to the coefficient of a Dirac delta, whose strength instead has energy-times-length dimensions before reduction.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.solidstate.electronic.kronig_penney.delta_comb import (
    DimensionlessKronigPenneyDeltaComb,
)
from projectkoios.physkit.units import ScalarQuantity, Unitless, VectorQuantity

In [ ]:
barrier_strength = 4.0
phase_parameters = VectorQuantity(
    magnitude=np.linspace(0.001, 4.0 * np.pi, 20000),
    unit=Unitless(),
)
model = DimensionlessKronigPenneyDeltaComb(
    barrier_strength=ScalarQuantity(
        magnitude=barrier_strength,
        unit=Unitless(),
    )
)
dispersion_values = model.dispersion_values(phase_parameters=phase_parameters)
sampling = model.sample_allowed_bands(phase_parameters=phase_parameters)

In [ ]:
retained_dispersion_values = model.dispersion_values(
    phase_parameters=sampling.allowed_phase_parameters
)
assert np.all(np.abs(retained_dispersion_values.magnitude) <= 1.0)
assert np.allclose(
    np.cos(sampling.bloch_phases.magnitude),
    retained_dispersion_values.magnitude,
    rtol=0.0,
    atol=5.0e-15,
)
assert (
    sampling.allowed_phase_parameters.magnitude.size < phase_parameters.magnitude.size
)
assert np.all(
    sampling.reduced_energies.magnitude[1:] > sampling.reduced_energies.magnitude[:-1]
)
sampling.allowed_phase_parameters.magnitude.size

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13.0, 5.0))
axes[0].plot(
    phase_parameters.magnitude / np.pi,
    dispersion_values.magnitude,
    label=r"$\cos z+P\sin(z)/z$",
)
axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1.0)
axes[0].axhline(-1.0, color="black", linestyle="--", linewidth=1.0)
axes[0].scatter(
    sampling.allowed_phase_parameters.magnitude / np.pi,
    retained_dispersion_values.magnitude,
    s=2,
    label="allowed samples",
    zorder=3,
)
axes[0].set(
    xlim=(0.0, 4.0),
    ylim=(-3.0, 3.0),
    xlabel=r"free phase $z/\pi$",
    ylabel="dispersion right-hand side",
    title="Allowed-value condition",
)
axes[0].grid(True)
axes[0].legend()

reduced_bloch_phases = sampling.bloch_phases.magnitude / np.pi
axes[1].scatter(
    reduced_bloch_phases,
    sampling.reduced_energies.magnitude,
    s=2,
    color="tab:blue",
)
axes[1].scatter(
    -reduced_bloch_phases,
    sampling.reduced_energies.magnitude,
    s=2,
    color="tab:blue",
)
axes[1].set(
    xlim=(-1.0, 1.0),
    xlabel=r"Bloch phase $ka/\pi$",
    ylabel=r"reduced energy $E/E_a$",
    title="Delta-comb band structure",
)
axes[1].grid(True)

figure.tight_layout()
plt.show()

## Interpretation

The horizontal bounds in the first panel divide allowed phase parameters from forbidden gaps. Mapping each allowed value through $ka=\arccos(f)$ produces the nonnegative first-zone branch; reflection supplies the negative branch because this model is inversion symmetric.

The graph is a calculated result of the stated dimensionless delta-comb model. It is not a material band structure. The earlier exploratory notebook treated a quantity labelled as an energy barrier as though it were a delta strength and selected points by an arbitrary residual tolerance. Neither behavior is retained.

## Exercises

1. Set $P=0$ and explain the folded free-electron branches.
2. Compare several positive barrier strengths without changing the energy sampling.
3. Derive the relation between a dimensional delta strength and $P$.
4. Locate band edges by root finding and compare them with the sampled graph.
